# LeafCheck: review and defend the frozen final CNN

This notebook explains and inspects the **completed saved model**. It does not retrain, select a new checkpoint or reevaluate the whole test set. Run it only after the final artifact has been frozen and evaluated. Use the project `.venv` as the notebook kernel.

Implementation and execution were assisted by Codex at the user's request. Review the methods and practice explaining them in your own words; do not claim independent authorship of assisted work.

## 1. Find the saved evidence

The artifact pairs the model weights with its exact class order, preprocessing and evaluation. The hashes identify the delivered version. A source checkout needs the separately restored artifact folder.

In [ ]:
from pathlib import Path
import json
import csv
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "leafcheck").is_dir() and (ROOT.parent / "leafcheck").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "leafcheck").is_dir(), "Open this notebook from the ML-Final-Project folder."
sys.path.insert(0, str(ROOT))
ARTIFACT = ROOT / "artifacts/leafcheck177-v1"
def read(name):
    return json.loads((ARTIFACT / name).read_text())
manifest = read("artifact.json")
assert manifest["final_evaluation_complete"]
assert manifest["supplementary_evaluation_complete"]
freeze = read("freeze.json")
print("Model:", freeze["model_version"])
print("Checkpoint SHA-256:", freeze["checkpoint_sha256"])
print("Dataset fingerprint:", freeze["dataset_fingerprint"])
print("Selection:", freeze["rationale"])

## 2. Explain the architecture and label mapping

The architecture is implemented with standard PyTorch layers and trained from random initialization. This is a custom CNN; it does not require manually implementing convolution or automatic differentiation. Residual shortcuts are an established design idea, not a novel invention.

**Checkpoint:** Why are there 177 output logits? Why must the mapping order match the classifier weights?

In [ ]:
import torch
from leafcheck.inference import Predictor

torch.set_num_threads(4)
predictor = Predictor(ARTIFACT, "cpu")
print(predictor.model)
print("Trainable parameters:", sum(p.numel() for p in predictor.model.parameters() if p.requires_grad))
print("Evaluation mode:", not predictor.model.training)
print("First five labels:", predictor.classes[:5])
print("Class count:", len(predictor.classes))

## 3. Trace tensor shapes

A convolution's output width is `floor((W + 2P - K) / S) + 1`. With width 128, kernel 3, padding 1 and stride 2, the result is 64. For a 160-pixel input, the same calculation gives 80. Global average pooling averages each channel across space; it avoids a large flattened classifier.

This zero input checks shapes only. It is not a disease example or evidence of accuracy.

In [ ]:
size = predictor.spec["input_size"]
x = torch.zeros(1, 3, size, size)
with torch.inference_mode():
    print("Input", tuple(x.shape))
    for index, block in enumerate(predictor.model.features):
        x = block(x)
        print("Feature stage", index, tuple(x.shape))
    logits = predictor.model.classifier(x)
    print("Logits", tuple(logits.shape))
assert logits.shape == (1, 177)


## 4. Explain preprocessing and augmentation

Inference selects frame zero, corrects EXIF orientation, converts to RGB, resizes with bilinear interpolation and gray padding, divides by 255, then standardizes with training-only statistics. No augmentation is applied at inference.

**Checkpoint:** Why are statistics fitted only on training data? How could hue shifts or tight crops remove diagnostic information?

In [ ]:
print(json.dumps(predictor.spec, indent=2))
config = read("config.json")
print("Training augmentation:", config["augmentation"])
print("Class-weight power:", config["class_weight_power"])

## 5. Compare completed development experiments

Validation selects checkpoints and methods. It is separate from the frozen test evaluation. The decision register states which changes helped and which were rejected. A repeated seed is a stability check, not statistical significance.

In [ ]:
import pandas as pd
experiments = pd.read_csv(ROOT / "docs/experiments.csv")
display(experiments[["run", "completed_epochs", "selected_epoch", "validation_accuracy", "validation_macro_f1", "training_seconds", "decision"]])
from IPython.display import Image as DisplayImage, display
display(DisplayImage(filename=str(ROOT / "docs/figures/experiment_comparison.png")))
display(DisplayImage(filename=str(ARTIFACT / "learning_curves.png")))

## 6. Read the final evaluation honestly

Accuracy is correct images divided by all images. Precision measures how often predictions of a class are correct; recall measures how many actual members of that class are found. F1 combines precision and recall. Macro-F1 weights each of the 177 labels equally. Rare classes have unstable estimates.

These are saved final results. Do not use the test mistakes to choose another model, learning rate or threshold.

In [ ]:
test = read("test_metrics.json")
rule = read("baseline_metrics.json")
reference = read("reference/test_metrics.json")
display(pd.DataFrame([
    {"method": "Final custom CNN", **{k:test[k] for k in ["correct", "sample_count", "accuracy", "macro_f1"]}},
    {"method": "Compact177 full-data reference", **{k:reference[k] for k in ["correct", "sample_count", "accuracy", "macro_f1"]}},
    {"method": "Most frequent training label", **{k:rule[k] for k in ["correct", "sample_count", "accuracy", "macro_f1"]}},
]))
per_class = pd.DataFrame(test["per_class"])
print("Classes with fewer than 10 test images:", (per_class.support < 10).sum())
display(per_class.sort_values(["support", "class_name"]).head(12))
display(DisplayImage(filename=str(ARTIFACT / "major_confusions.png")))

## 7. Keep the supplementary protocol distinct

The 183 screened PlantDoc photographs cover only three tomato labels. Predictions still compete among all 177 outputs. PlantDoc appears elsewhere in the master, and generic `Tomato leaf` is a healthy proxy. Its three-label macro-F1 has a different denominator from the 177-label score.

In [ ]:
external = read("external_metrics.json")
print({k:external[k] for k in ["correct", "sample_count", "accuracy", "macro_f1"]})
print("Conditional tomato rule:", external["conditional_rule"])
display(pd.DataFrame(external["per_class"]))
display(DisplayImage(filename=str(ARTIFACT / "external_confusion_matrix.png")))

## 8. Predict the saved success and mistake

These photographs were selected after final evaluation for the presentation. The interface uses the same runtime. Softmax outputs sum to one, but a large score does not prove correctness and does not establish a definitive diagnosis.

**Checkpoint:** What does the confidently wrong example teach you? Why do the historical pilots and Create ML not provide a fair same-test ranking?

In [ ]:
examples = read("demo_examples.json")
for row in examples:
    path = ARTIFACT / row["sample_file"]
    result = predictor.predict(path)
    assert result["label"] == row["predicted_class"]
    print(row["purpose"].upper(), "| reference:", row["actual_class"], "| prediction:", result["label"], "| score:", round(result["score"], 4))
    display(DisplayImage(filename=str(path), width=300))
    print("Top 3:", sorted(result["class_scores"].items(), key=lambda item:-item[1])[:3])

## 9. Rehearse the explanation

Use `docs/final_model_report.md`, `docs/model_defense_guide.md` and `docs/contributions.md` as evidence, and the project README for exact launch/reproduction commands.

Be ready to explain: the label space; why grouped splits matter; the two architecture shapes; why random weights satisfy the project's custom-model definition; what changed in each experiment; why validation macro-F1 selected this checkpoint; why overall accuracy hides weak rare labels; what the supplementary set proves and does not prove; what the website confidence score means; and which work was assisted.

A sensible next improvement should follow from the recorded weaknesses, such as independently collected, verified examples for rare labels. Propose it as future work without retraining against the already inspected test set.